In [0]:
# =============================================================================
# Notebook 07 — Target Encoding Inside CV + Improved Baseline
# Objective: Replace frequency-rank encoding of categorical experimental
#            variables with target encoding computed INSIDE each CV fold
#            (fit on train split only, applied to val split — no leakage).
#            Also adds smoothed target encoding for robustness on rare
#            categories. Compare: freq-encoding (NB06) vs target-encoding (NB07).
#
# Why negative R² in NB06:
#   Frequency-rank encoding discards the physical relationship between
#   categories and the target. "Pt cocatalyst" and "no cocatalyst" have
#   opposite effects on H2 rate, but their frequency ranks carry none of
#   that signal. Target encoding recovers it — at the cost of requiring
#   strict within-fold computation to avoid leakage.
#
# Inputs : ml_perovskites.X_exp_clean  (L3, contains freq-encoded cols)
#          ml_perovskites.X_mat_clean  (L1, material-only — no categoricals)
#          ml_perovskites.y_targets
# Outputs: ml_perovskites.cv_results_te  (Delta table)
#          /tmp/07_cv_comparison.png
#          /tmp/07_parity_best.png
# Seed   : 42
# Author : Bernardo Araldi da Silva | UFSC | 2026
# =============================================================================

# =============================================================================
# SECTION 0 — Install dependencies (run this cell alone first)
# =============================================================================
%pip install xgboost --quiet

# =============================================================================
# SECTION 1 — Imports and configuration
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import warnings
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.impute import SimpleImputer
import xgboost as xgb
from pyspark.sql import functions as F

warnings.filterwarnings("ignore")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
N_SPLITS = 5

# Smoothing factor for target encoding: m controls how much a rare category
# is pulled toward the global mean. Higher m → more regularization.
# m = 10 is a common default (equivalent to "at least 10 observations" weight).
TARGET_ENC_SMOOTHING = 10.0

print("Section 1 complete: imports and configuration loaded.")

# =============================================================================
# SECTION 2 — Load tables
# =============================================================================

print("\n--- Loading tables ---")

y_pd    = spark.table("ml_perovskites.y_targets").toPandas()
xl3_pd  = spark.table("ml_perovskites.X_exp_clean").toPandas()
xl1_pd  = spark.table("ml_perovskites.X_mat_clean").toPandas()
master_pd = spark.table("ml_perovskites.master").toPandas()

print(f"y_targets shape   : {y_pd.shape}")
print(f"X_exp_clean (L3)  : {xl3_pd.shape}")
print(f"X_mat_clean (L1)  : {xl1_pd.shape}")

# =============================================================================
# SECTION 3 — Identify categorical columns and rebuild raw categoricals
# =============================================================================

# The freq-encoded columns (exp_*_freq) in X_exp_clean need to be replaced
# with target-encoded versions computed per fold. We rebuild the raw
# categorical values from master and handle encoding inside the CV loop.

CATEGORICAL_COLS_RAW = [
    "cocatalyst_class",
    "scavenger_class",
    "run_type",
    "prep_class",
    "rate_confidence",
    "cutoff_status",
]

# Pull raw categoricals from master, aligned to source_row_id
cat_raw = master_pd[["source_row_id"] + CATEGORICAL_COLS_RAW].copy()
for col in CATEGORICAL_COLS_RAW:
    cat_raw[col] = (cat_raw[col]
                    .fillna("unknown")
                    .astype(str).str.strip().str.lower())

print(f"\nRaw categorical columns: {CATEGORICAL_COLS_RAW}")
for col in CATEGORICAL_COLS_RAW:
    print(f"  {col}: {cat_raw[col].nunique()} unique values")

# =============================================================================
# SECTION 4 — Aligned arrays
# =============================================================================

print("\n--- Preparing aligned arrays ---")

META = {"source_row_id", "doi", "base_compound", "family_proposed",
        "formula_reduced", "n_elements", "n_A_elements", "n_B_elements",
        "A_site_elements", "B_site_elements", "parse_ok", "Compound_original", "domain"}

# Freq-encoded categorical columns to REPLACE with target-encoded versions
FREQ_COLS = [c for c in xl3_pd.columns if c.endswith("_freq")]

# Numeric-only material features (L1)
feat_cols_l1 = [c for c in xl1_pd.columns if c not in META]

# L3 = L1 + numeric experimental cols (no freq cols — those will be target-encoded)
NUMERIC_EXP_COLS = [c for c in xl3_pd.columns
                    if c.startswith("exp_") and not c.endswith("_freq")]
feat_cols_l3_numeric = feat_cols_l1 + NUMERIC_EXP_COLS

# Alignment
common_ids = (set(y_pd["source_row_id"])
              & set(xl3_pd["source_row_id"])
              & set(xl1_pd["source_row_id"]))

y_al  = y_pd[y_pd["source_row_id"].isin(common_ids)].sort_values("source_row_id").reset_index(drop=True)
xl3_al = xl3_pd[xl3_pd["source_row_id"].isin(common_ids)].sort_values("source_row_id").reset_index(drop=True)
xl1_al = xl1_pd[xl1_pd["source_row_id"].isin(common_ids)].sort_values("source_row_id").reset_index(drop=True)
cat_al = cat_raw[cat_raw["source_row_id"].isin(common_ids)].sort_values("source_row_id").reset_index(drop=True)

y_log   = y_al["log1p_h2_rate"].values.astype(float)
groups  = y_al["doi"].values
ids     = y_al["source_row_id"].values

X_l1 = xl1_al[feat_cols_l1].values.astype(float)
X_l3_numeric = xl3_al[feat_cols_l3_numeric].values.astype(float)
# cat_al[CATEGORICAL_COLS_RAW] is used per-fold for target encoding

print(f"X_l1 (material only)         : {X_l1.shape}")
print(f"X_l3 numeric (no cat yet)     : {X_l3_numeric.shape}")
print(f"Categorical cols for TE       : {len(CATEGORICAL_COLS_RAW)}")
print(f"y shape                        : {y_log.shape}")
print(f"Unique DOI groups              : {len(np.unique(groups))}")

# =============================================================================
# SECTION 5 — Smoothed target encoding function
# =============================================================================

def target_encode_fold(cat_train_df, y_train, cat_val_df, cols, m=10.0):
    """
    Fit target encoding on (cat_train_df, y_train), apply to cat_val_df.
    Smoothing formula: enc = (n_cat * mean_cat + m * global_mean) / (n_cat + m)
    where n_cat = count of category in training set.
    Unknown categories in validation → global_mean.

    Returns numpy arrays: (encoded_train, encoded_val), shape (n_train, len(cols)) each.
    """
    global_mean = y_train.mean()
    enc_train   = np.zeros((len(cat_train_df), len(cols)), dtype=float)
    enc_val     = np.zeros((len(cat_val_df),   len(cols)), dtype=float)

    for j, col in enumerate(cols):
        train_vals = cat_train_df[col].values
        val_vals   = cat_val_df[col].values

        # Compute per-category statistics on training data
        cat_stats = {}
        unique_cats = np.unique(train_vals)
        for cat in unique_cats:
            mask = train_vals == cat
            n_cat   = mask.sum()
            mean_cat = y_train[mask].mean()
            # Smoothed estimate
            smoothed = (n_cat * mean_cat + m * global_mean) / (n_cat + m)
            cat_stats[cat] = smoothed

        # Encode training set (use leave-one-out to avoid self-leakage within fold)
        # For simplicity and safety, use smoothed mean without LOO (standard practice
        # when groups already prevent paper-level leakage via GroupKFold).
        enc_train[:, j] = np.array([cat_stats.get(v, global_mean) for v in train_vals])

        # Encode validation set (unknown → global_mean)
        enc_val[:, j]   = np.array([cat_stats.get(v, global_mean) for v in val_vals])

    return enc_train, enc_val


# =============================================================================
# SECTION 6 — Model definitions
# =============================================================================

def make_rf():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model",   RandomForestRegressor(
            n_estimators=400, max_features=0.33, min_samples_leaf=3,
            n_jobs=-1, random_state=RANDOM_SEED)),
    ])

def make_xgb():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model",   xgb.XGBRegressor(
            n_estimators=500, learning_rate=0.04, max_depth=4,
            subsample=0.8, colsample_bytree=0.7,
            reg_alpha=0.5, reg_lambda=2.0,
            random_state=RANDOM_SEED, verbosity=0, n_jobs=-1)),
    ])

def make_ridge():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler",  StandardScaler()),
        ("model",   Ridge(alpha=10.0, random_state=RANDOM_SEED)),
    ])

# =============================================================================
# SECTION 7 — Cross-validation with target encoding
# =============================================================================

print("\n--- Running GroupKFold CV with target encoding ---")
print(f"Smoothing parameter m = {TARGET_ENC_SMOOTHING}")
print()

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def metrics_dict(y_true, y_pred_log, fold, model_name, layer):
    r2_log   = r2_score(y_true, y_pred_log)
    rmse_log = rmse(y_true, y_pred_log)
    mae_log  = mean_absolute_error(y_true, y_pred_log)
    y_to   = np.expm1(y_true)
    y_po   = np.expm1(np.clip(y_pred_log, 0, None))
    r2_orig   = r2_score(y_to, y_po)
    rmse_orig = rmse(y_to, y_po)
    return {"fold": fold, "model": model_name, "layer": layer,
            "r2_log": r2_log, "rmse_log": rmse_log, "mae_log": mae_log,
            "r2_orig": r2_orig, "rmse_orig": rmse_orig,
            "n_val": len(y_true)}

gkf = GroupKFold(n_splits=N_SPLITS)

# Experiments:
# L1 = material-only (no target encoding needed — no categoricals)
# L3_te = material + numeric experimental + target-encoded categoricals
experiments = [
    ("Ridge",   "L1",    make_ridge, "l1"),
    ("Ridge",   "L3_te", make_ridge, "l3_te"),
    ("RF",      "L1",    make_rf,    "l1"),
    ("RF",      "L3_te", make_rf,    "l3_te"),
    ("XGBoost", "L1",    make_xgb,   "l1"),
    ("XGBoost", "L3_te", make_xgb,   "l3_te"),
]

all_results = []
oof_preds   = {}

for model_name, layer, make_fn, layer_key in experiments:
    oof = np.full(len(y_log), np.nan)

    for fold_idx, (train_idx, val_idx) in enumerate(gkf.split(X_l1, groups=groups)):
        y_train, y_val = y_log[train_idx], y_log[val_idx]

        if layer_key == "l1":
            X_train = X_l1[train_idx]
            X_val   = X_l1[val_idx]
        else:  # l3_te: material + numeric exp + target-encoded categoricals
            cat_train = cat_al.iloc[train_idx][CATEGORICAL_COLS_RAW]
            cat_val   = cat_al.iloc[val_idx][CATEGORICAL_COLS_RAW]

            te_train, te_val = target_encode_fold(
                cat_train, y_train, cat_val,
                CATEGORICAL_COLS_RAW, m=TARGET_ENC_SMOOTHING
            )

            X_train = np.hstack([X_l3_numeric[train_idx], te_train])
            X_val   = np.hstack([X_l3_numeric[val_idx],   te_val])

        pipe = make_fn()
        pipe.fit(X_train, y_train)
        y_pred = pipe.predict(X_val)
        oof[val_idx] = y_pred

        all_results.append(
            metrics_dict(y_val, y_pred, fold_idx + 1, model_name, layer)
        )

    oof_preds[(model_name, layer)] = oof
    fold_r2 = [r["r2_log"] for r in all_results
               if r["model"] == model_name and r["layer"] == layer]
    mean_r2 = np.mean(fold_r2)
    std_r2  = np.std(fold_r2)
    print(f"{model_name:8s} | {layer:6s} | R²(log)={mean_r2:+.3f}±{std_r2:.3f}")

results_df = pd.DataFrame(all_results)

# =============================================================================
# SECTION 8 — Summary table
# =============================================================================

print("\n--- Summary ---")

summary = (results_df
           .groupby(["model", "layer"])
           .agg(
               r2_log_mean= ("r2_log",   "mean"),
               r2_log_std=  ("r2_log",   "std"),
               rmse_log=    ("rmse_log", "mean"),
               mae_log=     ("mae_log",  "mean"),
               r2_orig=     ("r2_orig",  "mean"),
           )
           .reset_index()
           .sort_values("r2_log_mean", ascending=False))

print(summary.to_string(index=False))

best_l3 = summary[summary["layer"]=="L3_te"].iloc[0]
best_l1 = summary[summary["layer"]=="L1"].iloc[0]
te_gain = best_l3["r2_log_mean"] - best_l1["r2_log_mean"]

print(f"\nICC(DOI) ceiling (material-only): 0.58")
print(f"Best L1 model : {best_l1['model']}  R²={best_l1['r2_log_mean']:+.3f}")
print(f"Best L3_te    : {best_l3['model']}  R²={best_l3['r2_log_mean']:+.3f}")
print(f"Gain from TE  : {te_gain:+.3f} R² units (vs NB06 gain from freq-enc: +0.190)")

# =============================================================================
# SECTION 9 — Comparison plot: NB06 (freq-enc) vs NB07 (target-enc)
# =============================================================================

print("\n--- Generating plots ---")

nb06_r2 = {
    ("Ridge",   "L1"): -0.249, ("Ridge",   "L3"): -0.281,
    ("RF",      "L1"): -0.270, ("RF",      "L3"): -0.059,
    ("XGBoost", "L1"): -0.402, ("XGBoost", "L3"): -0.128,
}

models_order = ["Ridge", "RF", "XGBoost"]
layers_compare = [("L1","L1"), ("L3","L3_te")]  # (nb06 layer, nb07 layer)
colors = {"NB06_L1":"#aec6e8","NB06_L3":"#1f77b4","NB07_L1":"#f7c59f","NB07_L3":"#d62728"}

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(models_order))
bar_w = 0.20

nb07_l1 = summary[summary["layer"]=="L1"].set_index("model")["r2_log_mean"].to_dict()
nb07_l3 = summary[summary["layer"]=="L3_te"].set_index("model")["r2_log_mean"].to_dict()

offsets = [-1.5, -0.5, 0.5, 1.5]
labels_bars = ["NB06 L1 (freq)", "NB06 L3 (freq)", "NB07 L1", "NB07 L3 (TE)"]
bar_vals = [
    [nb06_r2.get((m,"L1"),   np.nan) for m in models_order],
    [nb06_r2.get((m,"L3"),   np.nan) for m in models_order],
    [nb07_l1.get(m, np.nan)           for m in models_order],
    [nb07_l3.get(m, np.nan)           for m in models_order],
]
bar_colors = ["#aec6e8","#1f77b4","#f7c59f","#d62728"]

for i, (vals, color, label) in enumerate(zip(bar_vals, bar_colors, labels_bars)):
    ax.bar(x + offsets[i]*bar_w, vals, bar_w, color=color, alpha=0.9, label=label)

ax.axhline(0.58, color="black", linestyle="--", linewidth=1.2, label="ICC ceiling (0.58)")
ax.axhline(0.0,  color="gray",  linestyle=":",  linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(models_order)
ax.set_ylabel("CV R² (log1p scale)")
ax.set_title("NB06 (frequency encoding) vs NB07 (target encoding inside folds)")
ax.legend(fontsize=8, ncol=2)
y_floor = min([v for vals in bar_vals for v in vals if not np.isnan(v)]) - 0.05
ax.set_ylim(bottom=y_floor, top=0.70)

plt.tight_layout()
plt.savefig("/tmp/07_cv_comparison.png", dpi=130, bbox_inches="tight")
plt.close()
print("  Saved: /tmp/07_cv_comparison.png")

# Parity plot — best model in NB07
best_model_name = best_l3["model"]
best_key = (best_model_name, "L3_te")
oof_best = oof_preds[best_key]
mask = ~np.isnan(oof_best)
r2_oof = r2_score(y_log[mask], oof_best[mask])

fig2, axes2 = plt.subplots(1, 2, figsize=(11, 5))

# Log scale
axes2[0].scatter(y_log[mask], oof_best[mask], alpha=0.3, s=8, color="#d62728")
lim = [y_log.min()-0.3, y_log.max()+0.3]
axes2[0].plot(lim, lim, "k--", linewidth=0.8)
axes2[0].set_xlim(lim); axes2[0].set_ylim(lim)
axes2[0].set_xlabel("Actual log1p(H₂ rate)")
axes2[0].set_ylabel("Predicted log1p(H₂ rate)")
axes2[0].set_title(f"OOF parity — {best_model_name} L3_te (R²={r2_oof:.3f})")

# Residuals vs actual
resid = oof_best[mask] - y_log[mask]
axes2[1].scatter(y_log[mask], resid, alpha=0.3, s=8, color="#d62728")
axes2[1].axhline(0, color="k", linestyle="--", linewidth=0.8)
axes2[1].set_xlabel("Actual log1p(H₂ rate)")
axes2[1].set_ylabel("Residual (pred − actual)")
axes2[1].set_title(f"Residuals — {best_model_name} L3_te")

plt.tight_layout()
plt.savefig("/tmp/07_parity_best.png", dpi=130, bbox_inches="tight")
plt.close()
print("  Saved: /tmp/07_parity_best.png")

# =============================================================================
# SECTION 10 — Write results
# =============================================================================

print("\n--- Writing cv_results_te to Delta table ---")

results_spark = spark.createDataFrame(
    results_df.where(pd.notnull(results_df), other=None)
)
(results_spark
    .write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("ml_perovskites.cv_results_te"))

n_cv = spark.table("ml_perovskites.cv_results_te").count()
print(f"Rows written: {n_cv}")

# =============================================================================
# SECTION 11 — Summary
# =============================================================================

print("\n" + "="*60)
print("NOTEBOOK 07 COMPLETE")
print("="*60)
print(f"Encoding strategy : smoothed target encoding (m={TARGET_ENC_SMOOTHING})")
print(f"                    computed INSIDE each CV fold — no leakage")
print(f"\nResults (mean CV R², log1p scale):")
for _, row in summary.iterrows():
    print(f"  {row['model']:8s} {row['layer']:6s}  "
          f"R²={row['r2_log_mean']:+.3f}±{row['r2_log_std']:.3f}  "
          f"RMSE={row['rmse_log']:.3f}")
print(f"\nICC(DOI) ceiling : 0.58")
print(f"Best model (L3_te): {best_l3['model']}  R²={best_l3['r2_log_mean']:+.3f}")
print(f"Gain from TE vs freq-enc: {te_gain:+.3f} R² units")
print(f"\nOutput table : ml_perovskites.cv_results_te ({n_cv} rows)")
print(f"\nNext step:")
if best_l3["r2_log_mean"] > 0.30:
    print(f"  R² > 0.30 — proceed to Notebook 08: Optuna HPO on {best_l3['model']}.")
elif best_l3["r2_log_mean"] > 0:
    print(f"  R² positive but below 0.30 — run Notebook 08 HPO.")
    print(f"  Also investigate: activity-tier stratification, two-stage model.")
else:
    print(f"  R² still negative after TE — deep diagnosis needed:")
    print(f"  Check feature-target correlation, potential labeling issues,")
    print(f"  or consider a two-stage hurdle model (classifier + regressor).")